# Customer Shopping Behavior: Data Preparation

Cleans and enriches the customer shopping dataset with pandas, then loads it into PostgreSQL so it can be visualized in Power BI (`dashboard/customer_behavior_analysis.pbix`).

**Steps:** load → explore → handle missing values → standardize column names → feature engineering → remove redundant column → load to PostgreSQL

> Place the dataset at `data/customer_shopping_behavior.csv` before running (see the project README).

In [ ]:
import os

import pandas as pd

## 1. Load data

In [ ]:
DATA_PATH = os.path.join("..", "data", "customer_shopping_behavior.csv")

df = pd.read_csv(DATA_PATH)
df.head()

## 2. Explore

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## 3. Missing values

Check for nulls, then fill missing review ratings with the median rating of the same product category.

In [ ]:
df.isnull().sum()

In [ ]:
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(
    lambda x: x.fillna(x.median())
)

df.isnull().sum()

## 4. Standardize column names

Lowercase, snake_case, and a cleaner name for the purchase amount column.

In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_")
df = df.rename(columns={"purchase_amount_(usd)": "purchase_amount"})

df.columns

## 5. Feature engineering

In [ ]:
# Age group: quartile-based buckets
labels = ["young adult", "adult", "middle-aged", "senior"]
df["age_group"] = pd.qcut(df["age"], q=4, labels=labels)

df[["age", "age_group"]].head(10)

In [ ]:
# Purchase frequency expressed in days.
# Keys are lowercase and matched case-insensitively so spelling variants
# such as "Bi-Weekly" / "Bi-weekly" map correctly instead of becoming NaN.
frequency_mapping = {
    "weekly": 7,
    "fortnightly": 14,
    "bi-weekly": 14,
    "monthly": 30,
    "quarterly": 90,
    "every 3 months": 90,
    "annually": 365,
}

df["purchase_frequency_days"] = (
    df["frequency_of_purchases"].str.strip().str.lower().map(frequency_mapping)
)

print("Unmapped frequency values:", df["purchase_frequency_days"].isnull().sum())
df[["purchase_frequency_days", "frequency_of_purchases"]].head(10)

## 6. Remove redundant column

`promo_code_used` and `discount_applied` should carry the same information. Verify, then drop one.

In [ ]:
df[["discount_applied", "promo_code_used"]].head(10)

In [ ]:
(df["discount_applied"] == df["promo_code_used"]).all()

In [ ]:
df = df.drop("promo_code_used", axis=1)

df.columns

## 7. Load into PostgreSQL (for Power BI)

Credentials are read from environment variables, never hardcoded. Set them before launching Jupyter:

```bash
export DB_PASSWORD="your-password"      # required
export DB_USER="postgres"               # optional (default: postgres)
export DB_HOST="localhost"              # optional
export DB_PORT="5432"                   # optional
export DB_NAME="customer_behavior"      # optional
```

On Windows PowerShell use `$env:DB_PASSWORD = "your-password"`. The database must already exist. If `DB_PASSWORD` is not set, this step is skipped.

In [ ]:
from urllib.parse import quote_plus

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "customer_behavior")
TABLE_NAME = "customer"

if not DB_PASSWORD:
    print("DB_PASSWORD is not set - skipping PostgreSQL load.")
else:
    from sqlalchemy import create_engine

    engine = create_engine(
        f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )
    df.to_sql(TABLE_NAME, engine, if_exists="replace", index=False)
    print(f"Loaded {len(df)} rows into table '{TABLE_NAME}' in database '{DB_NAME}'.")